# **＜重要＞本章で使用するファイル**

- daily_sales_data.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

In [ ]:
!pip install japanize_matplotlib

# Menu.89 探索的データ分析：仮説を立てて可視化しよう

生成されたコード例

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
import numpy as np

df = pd.read_csv("daily_sales_data.csv")

# 日付型に変換
df["date"] = pd.to_datetime(df["date"])
# 週末フラグ（仮説2用）
df["is_weekend"] = df["date"].dt.dayofweek >= 5 # 土日=True

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 要件1 広告費 × 売上（散布図 + 相関係数）
corr = df["ad_spend"].corr(df["sales"])
sns.scatterplot(data=df, x="ad_spend", y="sales", ax=axes[0], alpha=0.6)
axes[0].set_title("広告費と売上の関係")
axes[0].set_xlabel("広告費")
axes[0].set_ylabel("売上")
axes[0].text(0.05, 0.95, f"相関係数: {corr:.2f}", transform=axes[0].transAxes,verticalalignment="top")

# 要件2 売上の時系列（イベント日をハイライト）
axes[1].plot(df["date"], df["sales"], label="売上", color="blue")
# イベント日のみ赤で重ねる
event_df = df[df["event"] == 1]
axes[1].scatter(event_df["date"], event_df["sales"], color="red",
label="イベント日", s=15)
axes[1].set_title("売上の時系列（イベント日ハイライト）")
axes[1].set_xlabel("日付")
axes[1].set_ylabel("売上")
axes[1].legend()

# 要件3 天気別の売上（箱ひげ図）
sns.boxplot(data=df, x="weather", y="sales", ax=axes[2])
axes[2].set_title("天気別の売上分布")
axes[2].set_xlabel("天気")
axes[2].set_ylabel("売上")

# レイアウト調整 & 保存
plt.tight_layout()
plt.savefig("eda_analysis.png", dpi=300)
plt.close()

# Menu.90 ベースラインモデル：まず動かして、失敗から学ぼう

生成されたコード例

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

df = pd.read_csv("daily_sales_data.csv")
df["date"] = pd.to_datetime(df["date"])

# 時系列順にソート（念のため）
df = df.sort_values("date").reset_index(drop=True)

df_encoded = pd.get_dummies(df, columns=["weather", "event"],drop_first=True)

# 目的変数と説明変数
X = df_encoded.drop(columns=["sales", "date"])
y = df_encoded["sales"]

# 3. 学習データ / テストデータ分割（時系列）
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

date_test = df["date"].iloc[split_index:]

# 4. モデル学習 予測 & 評価
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== ベースラインモデル評価結果 ===")
print(f"MAE : {mae:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"R2 : {r2:.3f}")

# 5. 実測値 vs 予測値（時系列プロット）
plt.figure(figsize=(12, 5))
plt.plot(date_test, y_test.values, label="実際の売上", color="blue")
plt.plot(date_test, y_pred, label="予測売上（線形回帰）", color="red",
linestyle="--")
plt.title("実際の売上と予測売上の比較（ベースラインモデル）")
plt.xlabel("日付")
plt.ylabel("売上")
plt.legend()
plt.tight_layout()
plt.show()

# Menu.91 特徴量エンジニアリング：生成AIと対話して精度を上げよう

生成されたコード例

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

# データ読み込み & 前処理
df = pd.read_csv("daily_sales_data.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df_base = pd.get_dummies(df, columns=["weather", "event"], drop_first=True)

df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
df_improved = pd.get_dummies(df, columns=["weather", "event", "month","weekday"], drop_first=True)

# 学習 / テスト分割（時系列）
split_index = int(len(df) * 0.8)

def train_and_evaluate(df_encoded, label):
  X = df_encoded.drop(columns=["sales", "date"])
  y = df_encoded["sales"]

  X_train = X.iloc[:split_index]
  X_test = X.iloc[split_index:]
  y_train = y.iloc[:split_index]
  y_test = y.iloc[split_index:]

  model = LinearRegression()
  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)

  return {
    "label": label,
    "y_test": y_test,
    "y_pred": y_pred,
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred)
  }

# モデル学習 & 評価
result_base = train_and_evaluate(df_base, "改善前（時間特徴量なし）")
result_improved = train_and_evaluate(df_improved, "改善後（時間特徴量あり）")

# 評価結果の比較表示
print("=== モデル評価比較 ===")
for r in [result_base, result_improved]:
  print(f"\n【{r['label']}】")
  print(f"MAE : {r['MAE']:,.0f}")
  print(f"RMSE: {r['RMSE']:,.0f}")
  print(f"R2 : {r['R2']:.3f}")

# 予測結果の可視化
date_test = df["date"].iloc[split_index:]
plt.figure(figsize=(13, 5))
plt.plot(date_test, result_improved["y_test"].values, label="実際の売上",color="black")
plt.plot(date_test, result_base["y_pred"], label="予測（改善前）",linestyle="--", color="red")
plt.plot(date_test, result_improved["y_pred"], label="予測（改善後）",linestyle="--", color="blue")
plt.title("売上予測：改善前 vs 改善後（線形回帰）")
plt.xlabel("日付")
plt.ylabel("売上")
plt.legend()
plt.tight_layout()
plt.show()

# Menu.92 モデル選択：非線形モデルに挑戦しよう

生成されたコード例

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score

# データ読み込み & 前処理
df = pd.read_csv("daily_sales_data.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df["month"] = df["date"].dt.month
df["weekday"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["weekday"] >= 5).astype(int)

df_encoded = pd.get_dummies(
  df,
  columns=["weather", "event", "month", "weekday"],
  drop_first=True
)

X = df_encoded.drop(columns=["sales", "date"])
y = df_encoded["sales"]

# 時系列を考慮したデータ分割
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

date_test = df["date"].iloc[split_index:]

# モデル定義
models = {
  "LinearRegression": LinearRegression(),
  "RandomForest": RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
  )
}

results = {}

# 学習・評価
for name, model in models.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  results[name] = {
    "model": model,
    "y_pred": y_pred,
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred)
  }

# 評価結果の比較表
comparison_df = pd.DataFrame({
  "MAE": [results[m]["MAE"] for m in results],
  "RMSE": [results[m]["RMSE"] for m in results],
  "R2": [results[m]["R2"] for m in results]
}, index=results.keys())

print("=== モデル評価比較 ===")
print(comparison_df.round(3))

# 予測結果の可視化
plt.figure(figsize=(13, 5))

plt.plot(date_test, y_test.values,label="実際の売上", color="black")
plt.plot(date_test, results["LinearRegression"]["y_pred"],label="線形回帰 予測", linestyle="--", color="blue")
plt.plot(date_test, results["RandomForest"]["y_pred"],label="ランダムフォレスト 予測", linestyle="--", color="red")

plt.title("売上予測：線形回帰 vs ランダムフォレスト")
plt.xlabel("日付")
plt.ylabel("売上")
plt.legend()
plt.tight_layout()
plt.show()

特徴量重要度を確認する

In [ ]:
print(X.columns)
print(results["RandomForest"]["model"].feature_importances_)